In [0]:
import json
import pandas as pd
from pyspark.sql import functions as F

from databricks_langchain import ChatDatabricks
from langchain_core.tools import tool
from langchain.agents import create_agent
import mlflow

print("Imports complete")
print("Databricks user:", spark.sql("SELECT current_user()").first()[0])

DEMO_VERIFIED_MEMBER_ID = "MEM-0317"
MODEL_ENDPOINT = "databricks-gpt-oss-20b"

TABLES = {
    "claims": "claims.analytics.fact_claims",
    "members": "claims.analytics.dim_member",
    "policies": "claims.analytics.dim_policy",
    "plans": "claims.analytics.dim_plan",
    "hospitals": "claims.analytics.dim_hospital"
}

customer_service_llm = ChatDatabricks(endpoint=MODEL_ENDPOINT, temperature=0.1, max_tokens=1500)

@tool
def get_claim_details(claim_id: str) -> str:
    """Get claim status, dates, amounts, plan, and hospital for the verified demo member."""
    claim = spark.table(TABLES["claims"]).alias("f")
    member = spark.table(TABLES["members"]).alias("m")
    policy = spark.table(TABLES["policies"]).alias("p")
    plan = spark.table(TABLES["plans"]).alias("pl")
    hospital = spark.table(TABLES["hospitals"]).alias("h")

    rows = (claim
        .filter((F.col("f.claim_id") == claim_id) &
                (F.col("f.member_id") == DEMO_VERIFIED_MEMBER_ID))
        .join(member, F.col("f.member_sk") == F.col("m.member_sk"), "left")
        .join(policy, F.col("f.policy_sk") == F.col("p.policy_sk"), "left")
        .join(plan, F.col("f.plan_sk") == F.col("pl.plan_sk"), "left")
        .join(hospital, F.col("f.hospital_sk") == F.col("h.hospital_sk"), "left")
        .select(
            F.col("f.claim_id").alias("claim_id"),
            F.col("f.status").alias("status"),
            F.col("f.rejection_code").alias("rejection_code"),
            F.col("f.billed_amount").alias("billed_amount"),
            F.col("f.approved_amount").alias("approved_amount"),
            F.col("f.admission_date").alias("admission_date"),
            F.col("f.discharge_date").alias("discharge_date"),
            F.col("f.submitted_ts").alias("submitted_at"),
            F.col("f.settled_ts").alias("settled_at"),
            F.col("p.policy_id").alias("policy_id"),
            F.col("pl.plan_code").alias("plan_code"),
            F.col("pl.sum_insured").alias("sum_insured"),
            F.col("pl.room_rent_cap_per_day").alias("room_rent_cap_per_day"),
            F.col("h.name").alias("hospital_name"),
            F.col("h.city").alias("hospital_city"))
        .limit(1).collect())

    if not rows:
        return "No matching claim was found for the currently verified member. Do not reveal whether it exists for another member."
    return json.dumps(rows[0].asDict(), default=str)

@tool
def get_member_policies() -> str:
    """Get current policies and plan benefit fields for the verified demo member."""
    policy = spark.table(TABLES["policies"]).alias("p")
    plan = spark.table(TABLES["plans"]).alias("pl")
    rows = (policy
        .filter((F.col("p.member_id") == DEMO_VERIFIED_MEMBER_ID) &
                (F.col("p.is_current") == True))
        .join(plan, F.col("p.plan_code") == F.col("pl.plan_code"), "left")
        .select(
            F.col("p.policy_id").alias("policy_id"),
            F.col("p.plan_code").alias("plan_code"),
            F.col("p.valid_from").alias("valid_from"),
            F.col("p.valid_to").alias("valid_to"),
            F.col("pl.plan_family").alias("plan_family"),
            F.col("pl.sum_insured").alias("sum_insured"),
            F.col("pl.room_rent_cap_per_day").alias("room_rent_cap_per_day"),
            F.col("pl.copay_pct").alias("copay_pct"),
            F.col("pl.ped_waiting_months").alias("ped_waiting_months"),
            F.col("pl.initial_waiting_days").alias("initial_waiting_days"),
            F.col("pl.annual_premium").alias("annual_premium"))
        .limit(10).collect())
    if not rows:
        return "No current policy was found for the currently verified member."
    return json.dumps([r.asDict() for r in rows], default=str)


@tool
def get_hospital_info(hospital_id: str) -> str:
    """Look up a hospital's directory details using its hospital ID."""
    rows = (spark.table(TABLES["hospitals"])
        .filter(F.col("hospital_id") == hospital_id)
        .select("hospital_id", "name", "city", "city_tier", "state")
        .limit(1).collect())
    if not rows:
        return "No hospital was found with that hospital ID."
    return json.dumps(rows[0].asDict(), default=str)


@tool
def search_hospital_directory(hospital_name: str) -> str:
    """
    Search the internal hospital directory using a hospital name.
    The query may also include a city or state (e.g. "Fortis Escorts Jaipur").
    Returns matching hospital IDs, names, cities, and states.
    """
    tokens = [t for t in hospital_name.lower().split() if len(t) > 2]
    if not tokens:
        return "Please provide a more specific hospital name to search for."

    haystack = F.lower(F.concat_ws(" ", F.col("name"), F.col("city"), F.col("state")))
    condition = F.lit(True)
    for token in tokens:
        condition = condition & haystack.contains(token)

    rows = (
        spark.table(TABLES["hospitals"])
        .filter(condition)
        .select("hospital_id", "name", "city", "city_tier", "state")
        .limit(10)
        .collect()
    )

    if not rows:
        return "No matching hospital was found in the internal directory."

    return json.dumps(
        [row.asDict() for row in rows],
        default=str
    )

customer_service_tools = [
    get_claim_details,
    get_member_policies,
    get_hospital_info,
    search_hospital_directory
]


CUSTOMER_SERVICE_SYSTEM_PROMPT = f"""
You are a helpful customer service assistant for a health insurance claims service.
The application has verified the customer for this notebook session.
The only member whose private claim and policy data you may retrieve is {DEMO_VERIFIED_MEMBER_ID}.
The tools are bound to this verified context. Never try to select another member.

Rules:
- Use get_claim_details for claim-specific questions.
- Use get_member_policies for current policy and plan benefit questions.
- Use get_hospital_info for hospital directory questions when you already have a hospital ID.
- Use search_hospital_directory when the user provides a hospital name (optionally with city or state) and wants to find it in the internal directory.
- Never ask the user for a hospital ID when a name-based directory search can answer the question.
- If multiple hospitals match, return the candidates and distinguish them using their names and locations.
- Base factual answers on tool results. Say clearly when information is unavailable.
- Explain statuses without changing their meaning.
- Explain rejection codes only when the retrieved data supports it.
- Do not claim a claim is payable or covered unless retrieved information explicitly supports it.
- Do not provide medical advice or reveal another member's information.
- Never invent amounts, terms, dates, or hospital details.
- Keep responses concise and understandable.
"""

customer_service_agent = create_agent(
    model=customer_service_llm,
    tools=customer_service_tools,
    system_prompt=CUSTOMER_SERVICE_SYSTEM_PROMPT
)

Imports complete
Databricks user: kiritjain@cyntexa.com
